# AI Data Engineer Agent - Dataset Analysis Demo

This notebook demonstrates the automated dataset analysis capabilities of the AI Data Engineer Agent.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1. Generate Sample Dataset

Let's create a sample dataset to demonstrate the analysis capabilities.

In [ ]:
# Generate sample classification dataset
X, y = make_classification(
    n_samples=10000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    n_classes=2,
    random_state=42
)

# Create DataFrame
feature_names = [f'feature_{i}' for i in range(20)]
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

# Add some categorical features
df['category'] = np.random.choice(['A', 'B', 'C'], size=len(df))
df['region'] = np.random.choice(['North', 'South', 'East', 'West'], size=len(df))

# Introduce some missing values
missing_indices = np.random.choice(df.index, size=int(0.05 * len(df)), replace=False)
df.loc[missing_indices, 'feature_0'] = np.nan

# Add duplicates
duplicates = df.sample(n=100, random_state=42)
df = pd.concat([df, duplicates], ignore_index=True)

print(f"Dataset shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")
df.head()

## 2. Automated Dataset Analysis

Now let's simulate the AI-powered analysis that our system performs.

In [ ]:
def analyze_dataset(df):
    """Simulate AI-powered dataset analysis"""
    
    analysis = {
        'basic_stats': {
            'rows': len(df),
            'columns': len(df.columns),
            'memory_usage_mb': df.memory_usage(deep=True).sum() / 1024**2
        },
        'quality': {
            'missing_values': {
                'total_missing': df.isnull().sum().sum(),
                'missing_percentage': (df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100
            },
            'duplicates': {
                'duplicate_rows': df.duplicated().sum(),
                'duplicate_percentage': (df.duplicated().sum() / len(df)) * 100
            },
            'data_quality_score': 1 - (df.isnull().sum().sum() / (len(df) * len(df.columns)))
        },
        'columns': {},
        'target_suggestions': []
    }
    
    # Analyze each column
    for col in df.columns:
        col_analysis = {
            'type': str(df[col].dtype),
            'unique_values': df[col].nunique(),
            'null_percentage': (df[col].isnull().sum() / len(df)) * 100,
            'memory_usage': df[col].memory_usage(deep=True) / 1024**2
        }
        
        if df[col].dtype in ['int64', 'float64']:
            col_analysis.update({
                'mean': df[col].mean(),
                'std': df[col].std(),
                'min': df[col].min(),
                'max': df[col].max(),
                'skewness': df[col].skew()
            })
        
        analysis['columns'][col] = col_analysis
    
    # Suggest target variables
    for col in df.columns:
        if col == 'target':  # Skip if already named target
            continue
            
        unique_ratio = df[col].nunique() / len(df)
        if unique_ratio < 0.1:  # Likely categorical
            analysis['target_suggestions'].append({
                'column': col,
                'type': 'categorical',
                'score': min(5, int(5 * (1 - unique_ratio))),
                'reasons': ['Low unique ratio', 'Good for classification']
            })
    
    return analysis

# Perform analysis
analysis_results = analyze_dataset(df)
print("Dataset Analysis Complete!")
print(f"Quality Score: {analysis_results['quality']['data_quality_score']:.2f}")
print(f"Suggested Targets: {len(analysis_results['target_suggestions'])}")

## 3. Visualize Analysis Results

Create visualizations to better understand the dataset.

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Data quality overview
quality_data = [
    analysis_results['quality']['missing_values']['missing_percentage'],
    analysis_results['quality']['duplicates']['duplicate_percentage']
]
labels = ['Missing Values', 'Duplicates']
axes[0,0].bar(labels, quality_data, color=['red', 'orange'])
axes[0,0].set_title('Data Quality Issues (%)')
axes[0,0].set_ylabel('Percentage')

# 2. Column types distribution
column_types = {}
for col, info in analysis_results['columns'].items():
    dtype = info['type']
    column_types[dtype] = column_types.get(dtype, 0) + 1

axes[0,1].pie(column_types.values(), labels=column_types.keys(), autopct='%1.1f%%')
axes[0,1].set_title('Column Types Distribution')

# 3. Missing values by column
missing_by_col = [(col, info['null_percentage']) 
                  for col, info in analysis_results['columns'].items() 
                  if info['null_percentage'] > 0]
if missing_by_col:
    cols, missing_pct = zip(*missing_by_col)
    axes[1,0].barh(cols, missing_pct, color='red')
    axes[1,0].set_title('Missing Values by Column (%)')
    axes[1,0].set_xlabel('Percentage')
else:
    axes[1,0].text(0.5, 0.5, 'No Missing Values', ha='center', va='center', transform=axes[1,0].transAxes)
    axes[1,0].set_title('Missing Values by Column')

# 4. Target variable distribution
if 'target' in df.columns:
    target_counts = df['target'].value_counts()
    axes[1,1].pie(target_counts.values, labels=target_counts.index, autopct='%1.1f%%')
    axes[1,1].set_title('Target Variable Distribution')
else:
    axes[1,1].text(0.5, 0.5, 'No Target Variable', ha='center', va='center', transform=axes[1,1].transAxes)

plt.tight_layout()
plt.show()

## 4. Generate Data Cleaning Pipeline

Based on the analysis, generate an automated cleaning pipeline.

In [ ]:
def generate_cleaning_pipeline(analysis):
    """Generate automated cleaning pipeline based on analysis"""
    
    pipeline_steps = []
    
    # Remove duplicates
    if analysis['quality']['duplicates']['duplicate_percentage'] > 1:
        pipeline_steps.append({
            'step': 'remove_duplicates',
            'reason': f"Found {analysis['quality']['duplicates']['duplicate_percentage']:.1f}% duplicate rows",
            'code': "df = df.drop_duplicates()"
        })
    
    # Handle missing values
    if analysis['quality']['missing_values']['missing_percentage'] > 0:
        missing_cols = [col for col, info in analysis['columns'].items() 
                       if info['null_percentage'] > 0]
        
        for col in missing_cols:
            col_info = analysis['columns'][col]
            if col_info['type'] in ['int64', 'float64']:
                pipeline_steps.append({
                    'step': f'impute_{col}',
                    'reason': f"Numerical column '{col}' has {col_info['null_percentage']:.1f}% missing values",
                    'code': f"df['{col}'].fillna(df['{col}'].median(), inplace=True)"
                })
            else:
                pipeline_steps.append({
                    'step': f'impute_{col}',
                    'reason': f"Categorical column '{col}' has {col_info['null_percentage']:.1f}% missing values",
                    'code': f"df['{col}'].fillna(df['{col}'].mode()[0], inplace=True)"
                })
    
    return pipeline_steps

# Generate cleaning pipeline
cleaning_pipeline = generate_cleaning_pipeline(analysis_results)

print("Generated Cleaning Pipeline:")
print("=" * 50)
for i, step in enumerate(cleaning_pipeline, 1):
    print(f"{i}. {step['step']}")
    print(f"   Reason: {step['reason']}")
    print(f"   Code: {step['code']}")
    print()

## 5. Summary

This notebook demonstrated the AI Data Engineer Agent's automated dataset analysis capabilities:

✅ **Dataset Profiling**: Automatic detection of column types, missing values, and data quality issues

✅ **Quality Assessment**: Comprehensive quality scoring and issue identification

✅ **Target Variable Suggestions**: AI-powered identification of potential target variables

✅ **Automated Pipeline Generation**: Creation of data cleaning pipelines based on analysis

✅ **Visualization**: Rich visualizations for better understanding of the dataset

The next step would be to apply the generated cleaning pipeline and proceed with feature engineering and model training!